In [0]:
def validate_primary_key(
    df: DataFrame,
    primary_key: str | list[str]
) -> bool:
    """
    Valida se a chave primária é única.
    """

    if isinstance(primary_key, str):
        primary_key = [primary_key]

    duplicates = (
        df.groupBy(*primary_key)
        .agg(count("*").alias("TOTAL"))
        .filter(col("TOTAL") > 1)
        .count()
    )

    if duplicates > 0:
        raise ValueError(
            f"Validation failed: {duplicates} duplicated primary key(s) found."
        )

    print("✔ Primary key validation passed.")

    return True

In [0]:
def validate_not_null(
    df: DataFrame,
    columns: list[str]
) -> bool:
    """
    Valida se colunas obrigatórias possuem valores nulos.
    """

    for column in columns:

        nulls = (
            df.filter(col(column).isNull())
            .count()
        )

        if nulls > 0:
            raise ValueError(
                f"Validation failed: column '{column}' contains {nulls} null value(s)."
            )

    print("✔ Not Null validation passed.")

    return True

In [0]:
def validate_foreign_key(
    fact_df: DataFrame,
    dimension_df: DataFrame,
    key: str
) -> bool:
    """
    Valida integridade referencial entre duas tabelas.
    """

    invalid = (
        fact_df.select(key)
        .distinct()
        .join(
            dimension_df.select(key).distinct(),
            key,
            "leftanti"
        )
        .count()
    )

    if invalid > 0:
        raise ValueError(
            f"Validation failed: {invalid} foreign key(s) not found in dimension."
        )

    print("✔ Foreign key validation passed.")

    return True

In [0]:
def validate_years(
    df: DataFrame,
    year_column: str = "ANO_REFERENCIA"
) -> bool:
    """
    Valida se existe pelo menos um ano no DataFrame.
    """

    years = (
        df.select(year_column)
        .distinct()
        .count()
    )

    if years == 0:
        ###-----mensagem no telegram
        raise ValueError(
            "Validation failed: no reference years found."
        )

    print("✔ Year validation passed.")

    return True

In [0]:
def validate_schema(
    df: DataFrame,
    expected_schema: list[str]
) -> bool:
    """
    Valida se o schema do DataFrame corresponde ao esperado.
    """

    current_schema = df.columns

    missing = sorted(
        set(expected_schema) - set(current_schema)
    )

    unexpected = sorted(
        set(current_schema) - set(expected_schema)
    )

    if missing or unexpected:

        message = []

        if missing:
            message.append(f"Missing columns: {missing}")

        if unexpected:
            message.append(f"Unexpected columns: {unexpected}")

        raise ValueError(
            "Validation failed: " + " | ".join(message)
        )

    print("✔ Schema validation passed.")

    return True